In [ ]:
import utca
import osmnx as ox
import networkx as nx
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Patch
import mpltern

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    # Font
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Lines and markers
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    # Layout
    "figure.constrained_layout.use": True,
})

cm = 1 / 2.54

# districts of budapest on the symbolic plane

Calculate mosaic stats for all districts

In [ ]:
results = []

for kerulet in tqdm(range(1,24), desc='kerulet'):
    G = ox.load_graphml(f'output/bp_ker_simplified/{kerulet}.graphml')
    G = utca.prepare_graph(G)
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()
    stats = utca.graph_stats(G)
    stats.update({'kerulet': str(kerulet)})
    results.append(stats)

keruletek = pd.DataFrame(results)
keruletek.fillna(0, inplace=True)
keruletek['XYT'] = keruletek['X'] + keruletek['Y'] + keruletek['T']
keruletek['X^'] = keruletek['X'] / keruletek['XYT']
keruletek['Y^'] = keruletek['Y'] / keruletek['XYT']
keruletek['T^'] = keruletek['T'] / keruletek['XYT']

In [ ]:
#bp_palette = sns.color_palette('mako', 3)
bp_palette = [sns.color_palette('cool', 7)[x] for x in [0,3,6]][::-1]
bp_palette

In [ ]:
buda = [1,2,3,11,12,22]
colors = []
for i in range(1,24):
    if i in buda:
        colors.append(bp_palette[2])
    elif i == 21:#csepel
        colors.append(bp_palette[1])
    else:
        colors.append(bp_palette[0])

In [ ]:
buda_index = keruletek['kerulet'].astype(int).isin(buda)
pest_index = ~keruletek['kerulet'].astype(int).isin(buda+[21])
csepel_index = keruletek['kerulet'] == '21'
keruletek[pest_index]

Plot districts on the symbolic plane, highlighting Buda vs Pest

In [ ]:
fig = plt.figure(figsize=(14*cm,6*cm))
#fig.subplots_adjust(left=0.05, right=0.95)

ax = fig.add_subplot(121)

#ax.scatter("n", "v", s=80, c=colors, alpha=0.6, data=keruletek)
ax.scatter("n", "v", s=80, color=bp_palette[2], alpha=0.6, data=keruletek[buda_index], label='Buda')
ax.scatter("n", "v", s=80, color=bp_palette[0], alpha=0.6, data=keruletek[pest_index], label='Pest')
ax.scatter("n", "v", s=80, color=bp_palette[1], alpha=0.6, data=keruletek[csepel_index], label='Csepel')

#x1 = np.linspace(2,3,20)
#y1 = 2*x1/(x1-1)
#x3 = np.linspace(3,6,60)
#y3 = 2*x3/(x3-2)
#ax.plot(x1,y1, c='0.2')
#ax.plot([2,3],[4,6], c='0.2')
#ax.plot(x3,y3, c='0.2')
#ax.plot([3,6], [3,3], c='0.2')

for _, row in keruletek.iterrows():
    ax.annotate(
        str(row["kerulet"]),
        (row["n"], row["v"]),
        ha="center",
        va='center',
        fontsize=6
    )
ax.grid()

#ax.set_title("Districts of Budapest")
ax.set_xlabel("$\\overline{n}^*$")
ax.set_ylabel("$\\overline{v}^*$")#, rotation=0)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.25), ncol=3, frameon=True)

ax2 = fig.add_subplot(122, projection='ternary')

ax2.set_tlabel("$\\hat{Y}$")
ax2.set_llabel("$\\hat{T}$")
ax2.set_rlabel("$\\hat{X}$")
ax2.scatter("Y^", "T^", "X^", s=60, c=colors, alpha=0.6, data=keruletek)
for _, row in keruletek.iterrows():
    ax2.text(
        row["Y^"],
        row["T^"],
        row["X^"],
        row["kerulet"],
        ha="center",
        va='center',
        fontsize=4
    )

ax2.set_ternary_lim(
    0, 0.55,  # tmin, tmax
    0.45, 1,  # lmin, lmax
    0, 0.55,  # rmin, rmax
)

mode='horizontal'
#ax2.laxis.set_label_rotation_mode(mode)
#ax2.raxis.set_label_rotation_mode(mode)

ax2.grid()

plt.show()
#fig.savefig("output/figs_maj10/keruletek_tern.pdf")

# historical timeline

In [ ]:
import matplotlib.colors as mcolors
# Your bounds
vmin = 1850
vmax = 2025

# --- Truncate plasma to the useful range ---
base_cmap = plt.get_cmap('plasma')

# Use full plasma, but you *could* truncate if desired
colors = base_cmap(np.linspace(0, 1, 256))
cmap = mcolors.ListedColormap(colors)

# Set color for values below vmin
grey = '0.3'
cmap.set_under(grey)

# --- Normalization ---
norm = mcolors.Normalize(vmin=vmin, vmax=vmax, clip=False)

In [ ]:
def plot_map_timeline(gdf, ax, fig, last=True, title=None):
    #gdf = joined
    #gdf["date_num"] = gdf["date"].astype("int64")
    #gdf["date_num"] = gdf["date"].dt.strftime("%Y")

    #cmap = 'plasma'

    gdf.plot(
        ax=ax,
        column="year",
        cmap=cmap,
        norm=norm,
        linewidth=0.3,
        #alpha=0.8,
        legend=False,
        missing_kwds={'color': grey},
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if title is not None:
        title = f'District {title}'
    ax.set_ylabel(title)

    # create colorbar manually
    #norm = mpl.colors.Normalize(
    #    vmin=gdf["date_num"].min(),
    #    vmax=gdf["date_num"].max()
    #)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm._A = []

    if last:
        cbar = fig.colorbar(sm, ax=ax, shrink=0.8, orientation='horizontal')
        # format ticks back to dates
        #ticks = cbar.get_ticks()
        #tick_labels = pd.to_datetime(ticks).strftime("%Y")
        #cbar.set_ticks(ticks)
        #cbar.set_ticklabels(tick_labels)
        cbar.set_label("Year")
        na_patch = Patch(facecolor=grey, edgecolor='black', label='N/A')
        cbar.ax.legend(
            handles=[na_patch],
            loc='center left',
            bbox_to_anchor=(-0.7, 0),
            frameon=False,
            fontsize='small',
        )


In [ ]:
def plot_keruletek(erdekes):
    n = len(erdekes)
    hist_cache = utca.load_streets(query="cityname LIKE 'Budapest%'")
    fig, axs = plt.subplots(n, 3, figsize=(14*cm, n*4*cm))

    for n_row, kerulet in enumerate(tqdm(erdekes)):
        G = ox.load_graphml(f'output/bp_ker_simplified/{kerulet}.graphml')
        edges = ox.graph_to_gdfs(G, nodes=False)
        edges = utca.join_historical_streets(edges, hist_cache=hist_cache, fill_na=True)
        timeline = utca.get_timeline(edges)
        plot_map_timeline(edges, axs[n_row, 0], fig, (n_row==n-1), title=kerulet)
        axs[n_row, 1].plot('year', 'n', '', data=timeline, label='_')
        axs[n_row, 1].set_xlabel("year")
        axs[n_row, 1].set_ylabel("$\\overline{n}^*$")#, rotation=0)
        axs[n_row, 2].plot('year', 'v', '', data=timeline, label='_')
        axs[n_row, 2].set_xlabel("year")
        axs[n_row, 2].set_ylabel("$\\overline{v}^*$")#, rotation=0)
    return fig

In [ ]:
plot_keruletek([17, 19])

In [ ]:
fig = plot_keruletek([4, 11, 12, 19])

In [ ]:
#fig.savefig("output/figs_maj10/keruletek_vn_short.pdf")